# Khám phá Dữ liệu Chuyên sâu (EDA) - Sub-task 3.1
Notebook này thực hiện việc kết nối với Supabase, kéo dữ liệu từ View `mv_bi_mart_hourly_measures` kết hợp với bảng `dim_solar_site`, kiểm tra chất lượng và thống kê mô tả.

In [1]:
import os
import logging
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import urllib.parse 

# 1. Cấu hình Logging (Vẫn in ra console của Jupyter cho dễ theo dõi)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


# 2. Load biến môi trường và kết nối
load_dotenv()
db_user = os.getenv("DB_USER")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")
raw_password = os.getenv("DB_PASSWORD")
db_password = urllib.parse.quote_plus(raw_password) if raw_password else None

if not all([db_user, db_password, db_host, db_port, db_name]):
    logger.error("Không tìm thấy đủ các biến môi trường trong file .env!")
else:
    DATABASE_URL = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    try:
        engine = create_engine(DATABASE_URL)
        logger.info("Đã kết nối thành công với Supabase.")
    except Exception as e:
        logger.critical(f"Lỗi kết nối database: {e}")

2026-07-23 22:36:46 | INFO     | Đã kết nối thành công với Supabase.


In [2]:
# 3. Kéo dữ liệu từ BI Mart
query = """
    SELECT 
        m.*, 
        s.capacity_kw 
    FROM bi_mart.mv_bi_mart_hourly_measures m
    LEFT JOIN bi_mart.dim_solar_site s ON m.site_id = s.site_id
"""
logger.info("Đang thực thi truy vấn kéo dữ liệu...")
df = pd.read_sql_query(query, engine)

if df.empty:
    logger.error("View trả về 0 dòng! Cần kiểm tra lại DWH.")
else:
    logger.info(f"Đã kéo thành công {df.shape[0]} dòng và {df.shape[1]} cột.")
    
    # QA/QC Check
    missing_data = df.isnull().sum()
    total_missing = missing_data.sum()
    
    if total_missing == 0:
        logger.info("Chất lượng dữ liệu: ĐẠT.")
    else:
        logger.warning(f"Phát hiện {total_missing} giá trị bị thiếu!")
        display(pd.DataFrame(missing_data[missing_data > 0], columns=['Số lượng missing']))

2026-07-23 22:36:46 | INFO     | Đang thực thi truy vấn kéo dữ liệu...
2026-07-23 22:37:26 | INFO     | Đã kéo thành công 683665 dòng và 36 cột.
2026-07-23 22:37:27 | WARNING  | Phát hiện 1083824 giá trị bị thiếu!


,Số lượng missing
weather_type_id,108
shortwave_radiation,108
temperature_c,108
cloud_cover_total,108
cloud_cover_low,108
cloud_cover_mid,108
cloud_cover_high,108
diffuse_solar_radiation,108
direct_normal_irradiance,108
wind_speed,108


In [3]:
# 4. Thống kê mô tả toàn bộ
logger.info("BẢNG THỐNG KÊ MÔ TẢ TỔNG THỂ")

# Chọn cột số và tính thống kê
df_numeric = df.select_dtypes(include=['number'])
thong_ke_mo_ta = df_numeric.describe().T.round(2)

# Hiển thị bảng dạng HTML đẹp mắt trong Jupyter
display(thong_ke_mo_ta)

2026-07-23 22:37:27 | INFO     | BẢNG THỐNG KÊ MÔ TẢ TỔNG THỂ


,count,mean,std,min,25%,50%,75%,max
date_id,683665.0,20209218.44,6584.97,20200101.00,20201114.00,20210515.00,20211104.00,20220423.00
hourly_bucket,683665.0,11.48,6.93,0.00,5.00,11.00,17.00,23.00
site_id,683665.0,21.69,12.31,1.00,11.00,22.00,33.00,42.00
geo_id,683665.0,21.69,12.31,1.00,11.00,22.00,33.00,42.00
is_holiday,683665.0,0.00,0.06,0.00,0.00,0.00,0.00,1.00
is_semester,683665.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
is_exam,683665.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
weather_type_id,683557.0,5.58,3.52,1.00,2.00,6.00,8.00,22.00
shortwave_radiation,683557.0,185.77,268.34,0.00,0.00,10.00,322.00,1109.00
temperature_c,683557.0,14.78,6.22,-0.40,10.20,13.90,18.50,44.20


In [4]:
# 5. Thống kê tập trung vào site có capacity_kw
cap_col = 'capacity_kw' 

if cap_col in df.columns:
    df_capacity_not_null = df[df[cap_col].notnull()]
    
    if not df_capacity_not_null.empty:
        logger.info(f"BẢNG THỐNG KÊ MÔ TẢ (Chỉ tính {df_capacity_not_null.shape[0]} dòng có {cap_col})")
        
        df_not_null_numeric = df_capacity_not_null.select_dtypes(include=['number'])
        thong_ke_mo_ta_not_null = df_not_null_numeric.describe().T.round(2)
        
        display(thong_ke_mo_ta_not_null)
    else:
        logger.warning(f"Cảnh báo: Toàn bộ dữ liệu đều bị null ở cột {cap_col}!")
else:
    logger.error(f"Không tìm thấy cột '{cap_col}'!")

2026-07-23 22:37:29 | INFO     | BẢNG THỐNG KÊ MÔ TẢ (Chỉ tính 400377 dòng có capacity_kw)


,count,mean,std,min,25%,50%,75%,max
date_id,400377.0,20209411.52,6495.73,20200101.00,20201129.00,20210521.00,20211106.00,20220423.00
hourly_bucket,400377.0,11.48,6.93,0.00,5.00,11.00,17.00,23.00
site_id,400377.0,27.05,8.01,14.00,20.00,26.00,34.00,40.00
geo_id,400377.0,27.05,8.01,14.00,20.00,26.00,34.00,40.00
is_holiday,400377.0,0.00,0.06,0.00,0.00,0.00,0.00,1.00
is_semester,400377.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
is_exam,400377.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00
weather_type_id,400324.0,5.96,3.45,1.00,3.00,7.00,8.00,22.00
shortwave_radiation,400324.0,177.00,257.17,0.00,0.00,9.00,304.00,1081.00
temperature_c,400324.0,14.48,5.74,-0.40,10.40,13.60,17.90,40.80


In [5]:
# 3. Kéo dữ liệu từ BI Mart
query_weather = """
    SELECT timestamp, shortwave_radiation
    FROM staging.stg_open_meteo_weather_raw
"""
logger.info("Đang thực thi truy vấn kéo dữ liệu...")
df_weather = pd.read_sql_query(query_weather, engine)

if df_weather.empty:
    logger.error("View trả về 0 dòng! Cần kiểm tra lại DWH.")
else:
    logger.info(f"Đã kéo thành công {df_weather.shape[0]} dòng và {df_weather.shape[1]} cột.")
    
    # QA/QC Check
    missing_data = df_weather.isnull().sum()
    total_missing = missing_data.sum()
    
    if total_missing == 0:
        logger.info("Chất lượng dữ liệu: DAT.")
    else:
        logger.warning(f"Phat hien {total_missing} gia tri bi thieu!")
        display(pd.DataFrame(missing_data[missing_data > 0], columns=['So luong missing']))

    ## tinh trung binh va trung vi cua shortwave_radiation theo thoi gian trong ngay
    
    # Convert timestamp to datetime neu chua phai datetime
    df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'])
    
    df_weather['hour'] = df_weather['timestamp'].dt.hour
    hourly_avg = df_weather.groupby('hour')['shortwave_radiation'].mean()
    hourly_median = df_weather.groupby('hour')['shortwave_radiation'].median()
    
    print("Gia tri trung binh cua shortwave_radiation theo gio:")
    print(hourly_avg)
    print("\nGia tri trung vi cua shortwave_radiation theo gio:")
    print(hourly_median)

2026-07-23 22:37:30 | INFO     | Đang thực thi truy vấn kéo dữ liệu...
2026-07-23 22:37:37 | INFO     | Đã kéo thành công 850752 dòng và 2 cột.
2026-07-23 22:37:37 | INFO     | Chất lượng dữ liệu: DAT.


TypeError: dtype 'str' does not support operation 'mean'